In [ ]:
# ============================================================================
# CELL 1: SETUP & IMPORTS - PRODUCTION READY (UNBUFFERED LOGS)
# ============================================================================
import subprocess
import sys
import os
import random
import warnings
import math
import json
import csv
import time
import logging
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR, OneCycleLR

# --- FIX: Force Unbuffered Output (Crucial for debugging crashes) ---
os.environ['PYTHONUNBUFFERED'] = '1'

# --- AMP utilities ---
if int(torch.__version__.split('.')[0]) >= 2:
    from torch.amp import GradScaler, autocast
else:
    from torch.cuda.amp import GradScaler, autocast

# --- Optional libraries with fallbacks ---
try:
    import cv2
except Exception:
    cv2 = None
    print("[WARN] OpenCV (cv2) not available. Image/video processing will fail if used.")

try:
    from PIL import Image
except Exception:
    Image = None
    print("[WARN] PIL.Image not available. Image processing will fail if used.")

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None
    print("[WARN] Matplotlib not available. Visualization will fail if used.")

try:
    from tqdm.notebook import tqdm
except Exception:
    try:
        from tqdm import tqdm
        print("[INFO] Using standard tqdm fallback (non-notebook).")
    except Exception:
        tqdm = None
        print("[WARN] tqdm not available. Progress bars will not work.")

try:
    import seaborn as sns
except Exception:
    sns = None
    print("[WARN] Seaborn not available. Visualization may be limited.")

# --- TorchVision models & COCO utils ---
try:
    from torchvision.models.detection import (
        retinanet_resnet50_fpn_v2,
        RetinaNet_ResNet50_FPN_V2_Weights,
    )
    from torchvision.models.detection.retinanet import RetinaNetClassificationHead
except Exception:
    retinanet_resnet50_fpn_v2 = None
    RetinaNet_ResNet50_FPN_V2_Weights = None
    RetinaNetClassificationHead = None
    print("[WARN] torchvision.retinanet not available. Model training will fail if used.")

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except Exception:
    COCO = None
    COCOeval = None
    print("[WARN] pycocotools not available. COCO evaluation will fail if used.")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except Exception:
    A = None
    ToTensorV2 = None
    print("[WARN] albumentations not available. Data augmentation will fail if used.")
try:
    
    torch.multiprocessing.set_sharing_strategy('file_system')
except RuntimeError:
    pass
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Logger setup (Updated for instant output)
# ----------------------------------------------------------------------------
class ForceFlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

# Logger setup
LOG_PATH = WORK_DIR / "training.log"
root_logger = logging.getLogger()
for h in list(root_logger.handlers):
    root_logger.removeHandler(h)

file_handler = logging.FileHandler(str(LOG_PATH))
# Dùng handler mới này để log hiện ngay lập tức trên Kaggle console
stream_handler = ForceFlushHandler(sys.stdout) 

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] [%(levelname)s] %(message)s',
    handlers=[file_handler, stream_handler],
    force=True
)
logger = logging.getLogger(__name__)
logger.propagate = False

# ----------------------------------------------------------------------------
# Warnings
# ----------------------------------------------------------------------------
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torch.nn.parallel')

# ----------------------------------------------------------------------------
# Environment flags
# ----------------------------------------------------------------------------
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128')
os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '0')

# ----------------------------------------------------------------------------
# Device setup
# ----------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------------------------
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_seed(42, deterministic=False)

# ----------------------------------------------------------------------------
# TF32 / matmul optimization for P100
# ----------------------------------------------------------------------------
if torch.cuda.is_available():
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

# ----------------------------------------------------------------------------
# Runtime info logging
# ----------------------------------------------------------------------------
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        logger.info(f"CUDA version: {torch.version.cuda}")
        logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
        total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info(f"GPU Memory: {total_memory_gb:.2f} GB")
    except Exception:
        logger.warning("Unable to query full CUDA info.")
logger.info(f"Device set to: {device}")
logger.info(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
logger.info("Module availability summary:")
logger.info(f" OpenCV (cv2) available: {cv2 is not None}")
logger.info(f" PIL.Image available: {Image is not None}")
logger.info(f" Matplotlib available: {plt is not None}")
logger.info(f" tqdm available: {tqdm is not None}")
logger.info(f" Seaborn available: {sns is not None}")
logger.info(f" torchvision.retinanet available: {retinanet_resnet50_fpn_v2 is not None}")
logger.info(f" pycocotools available: {COCO is not None}")
logger.info(f" albumentations available: {A is not None}")
logger.info(f"Setup Complete. Device: {device}")
logger.info(f"Sharing Strategy: {torch.multiprocessing.get_sharing_strategy()}")
logger.info("All imports attempted")

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION - ADAMW STABLE MODE (OPTIMIZED & RESUME READY)
# ============================================================================
from typing import Dict, List
from pathlib import Path
import os

NUM_USER_CLASSES = 11
USER_CLASSES_LIST = [
    'Vehicle', 'Bus', 'Bicycle', 'Person', 'Engine', 'Truck',
    'Tricycle', 'Obstacle', 'Pothole', 'Traffic Light', 'Traffic Sign'
]

CONFIG: Dict = {
    # Dataset paths
    'TRAIN_ANN': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/annotations/instances_train.json'),
    'TRAIN_IMG': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/train'),
    'VAL_ANN': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/annotations/instances_val.json'),
    'VAL_IMG': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/val'),
    'TEST_ANN': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/annotations/instances_test.json'),
    'TEST_IMG': Path('/kaggle/input/traffic-object/ImBalanced-Data/ImBalanced-Data/test'),
    # Output & checkpoint
    'OUTPUT_DIR': Path('/kaggle/working'),
    'CHECKPOINT_DIR': Path('/kaggle/working/checkpoints'),
    
    # --- TRAINING PARAMS (FP16 + ADAMW) ---
    'EPOCHS': 50,
    'BATCH_SIZE': 8,         
    'ACCUMULATION_STEPS': 4, # Tương đương Batch 32
    'NUM_WORKERS': 2,
    'PIN_MEMORY': True,
    'PERSISTENT_WORKERS': True,
    'PREFETCH_FACTOR': 2,
    
    # --- OPTIMIZER ---
    'OPTIMIZER': 'AdamW',      
    'LEARNING_RATE': 1e-4,   
    'WEIGHT_DECAY': 1e-4,
    'WARMUP_EPOCHS': 1,
    
    # --- STABILITY ---
    'USE_AMP': True,         # Bật AMP cho nhanh (nếu NaN thì sửa thành False)
    'GRAD_CLIP': 0.1,        # Kẹp chặt gradient
    
    'FOCAL_ALPHA': 0.25,
    'FOCAL_GAMMA': 2.0,
    
    'MULTI_SCALE': True,
    'INPUT_SIZES': [384, 432, 480],
    'SCALE_CHANGE_FREQUENCY': 200,
    'BASE_INPUT_SIZE': 432,
    
    # --- DATA CLEANING ---
    'MIN_BOX_SIZE': 4.0,     
    'MIN_BOXES_PER_IMAGE': 1,
    
    # Validation
    'VAL_FREQUENCY': 2,      # <--- Đã sửa thành 1: Validate mỗi epoch
    'SCORE_THRESH': 0.05,
    'EARLY_STOPPING_PATIENCE': 5,
    'VAL_SUBSET': 1200,
    'SEED': 42,
    'LOG_BATCH_INTERVAL': 50,
    'LOG_LEVEL': 'INFO',
    'BEST_METRIC': 'mAP',
    
    # --- RESUME CONFIGURATION ---
    'RESUME_TRAINING': True, 
    # Đảm bảo bạn đã Add dataset chứa checkpoint vào notebook (tên là 'checkpoint')
    'RESUME_CHECKPOINT': '/kaggle/input/checkpoint-imbalance/checkpoints/latest.pth', 
    'RESUME_LOGS_DIR': '/kaggle/input/checkpoint-imbalance/logs',   
    
    'START_EPOCH': 0,
    'RUN_TEST_EVAL': True,
    'EXPORT_MODEL': True,
}

CONFIG['NUM_MODEL_CLASSES'] = len(USER_CLASSES_LIST) + 1
CONFIG['OUTPUT_DIR'].mkdir(parents=True, exist_ok=True)
CONFIG['CHECKPOINT_DIR'].mkdir(parents=True, exist_ok=True)
(CONFIG['OUTPUT_DIR'] / 'logs').mkdir(parents=True, exist_ok=True)
(CONFIG['OUTPUT_DIR'] / 'plots').mkdir(parents=True, exist_ok=True)

def validate_config(config: Dict):
    return config

CONFIG = validate_config(CONFIG)

logger.info("Configuration summary (ADAMW + AMP MODE)")
logger.info(f" Optimizer: {CONFIG['OPTIMIZER']} | LR: {CONFIG['LEARNING_RATE']}")
logger.info(f" Stability: Grad Clip = {CONFIG['GRAD_CLIP']} | AMP = {CONFIG['USE_AMP']}")
logger.info(f" Resume: {CONFIG['RESUME_TRAINING']} | Path: {CONFIG['RESUME_CHECKPOINT']}")

In [ ]:
# ============================================================================
# CELL 3: METRICS LOGGING (SYNTAX FIXED & RESUME READY)
# ============================================================================
import csv
import shutil
from typing import List, Dict, Any, Optional
from pathlib import Path
import pandas as pd
import torch
import matplotlib

# --- FIX: Force Matplotlib to non-interactive mode ---
try:
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
except Exception:
    plt = None

class MetricsLoggerV2:
    def __init__(self, config: Dict, logger):
        self.config = config
        self.logger = logger
        self.output_dir = Path(config['OUTPUT_DIR'])
        self.logs_dir = self.output_dir / "logs"
        self.plots_dir = self.output_dir / "plots"
        self.logs_dir.mkdir(parents=True, exist_ok=True)
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        
        self.epoch_metrics: List[Dict[str, Any]] = []
        self.val_metrics: List[Dict[str, Any]] = []
        
        # 1. Load history first (from Resume dir or Current dir)
        self.load_historical_data()
        
        # 2. Ensure local CSV files exist and contain the loaded history
        self._ensure_csv_files_sync()

    def load_historical_data(self):
        """
        Thông minh: Tự động tìm log cũ để load.
        Ưu tiên 1: File trong /kaggle/working/logs (nếu kernel restart nhẹ)
        Ưu tiên 2: File trong RESUME_LOGS_DIR (nếu start session mới từ input dataset)
        """
        if pd is None: return

        # Xác định nguồn log epoch
        epoch_source = self.logs_dir / "epoch_metrics.csv"
        if not epoch_source.exists() and self.config.get('RESUME_TRAINING') and self.config.get('RESUME_LOGS_DIR'):
            resume_dir = Path(self.config['RESUME_LOGS_DIR'])
            epoch_source = resume_dir / "epoch_metrics.csv"
            self.logger.info(f" Found RESUME log source: {epoch_source}")

        # Xác định nguồn log validation
        val_source = self.logs_dir / "validation_metrics.csv"
        if not val_source.exists() and self.config.get('RESUME_TRAINING') and self.config.get('RESUME_LOGS_DIR'):
            resume_dir = Path(self.config['RESUME_LOGS_DIR'])
            val_source = resume_dir / "validation_metrics.csv"

        # Load Epoch Data
        try:
            if epoch_source.exists():
                df = pd.read_csv(epoch_source)
                df = df.drop_duplicates(subset=['epoch'], keep='last').sort_values('epoch')
                self.epoch_metrics = df.to_dict('records')
                self.logger.info(f" Loaded {len(self.epoch_metrics)} epoch records from {epoch_source}")
        except Exception as e:
            self.logger.warning(f"Failed to load epoch logs: {e}")

        # Load Validation Data
        try:
            if val_source.exists():
                df = pd.read_csv(val_source)
                df = df.drop_duplicates(subset=['epoch'], keep='last').sort_values('epoch')
                self.val_metrics = df.to_dict('records')
                self.logger.info(f" Loaded {len(self.val_metrics)} validation records from {val_source}")
        except Exception as e:
            self.logger.warning(f"Failed to load validation logs: {e}")

    def _ensure_csv_files_sync(self):
        """Đồng bộ dữ liệu đã load vào file CSV ở working directory."""
        # Sync Epoch Metrics
        epoch_csv = self.logs_dir / "epoch_metrics.csv"
        fieldnames_epoch = [
            'epoch', 'train_loss', 'learning_rate', 'epoch_time_sec',
            'throughput', 'gpu_memory_mb', 'grad_norm', 'skip_count'
        ]
        
        with open(epoch_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames_epoch)
            writer.writeheader()
            if self.epoch_metrics:
                for row in self.epoch_metrics:
                    filtered_row = {k: row.get(k, 0) for k in fieldnames_epoch}
                    writer.writerow(filtered_row)

        # Sync Validation Metrics
        val_csv = self.logs_dir / "validation_metrics.csv"
        fieldnames_val = [
            'epoch', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium',
            'AP_large', 'precision', 'recall', 'f1', 'val_time_sec'
        ]
        
        with open(val_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames_val)
            writer.writeheader()
            if self.val_metrics:
                for row in self.val_metrics:
                    filtered_row = {k: row.get(k, 0) for k in fieldnames_val}
                    writer.writerow(filtered_row)
        
        self.logger.info(" History synchronized to local working CSVs.")

    def log_epoch(self, epoch: int, metrics: Dict[str, Any]):
        try:
            throughput = float(metrics.get('throughput', 0.0))
            gpu_memory_mb = float(metrics.get('gpu_memory_mb', 0.0))
            if torch.cuda.is_available() and gpu_memory_mb == 0.0:
                gpu_memory_mb = torch.cuda.memory_allocated() / 1024**2
            
            row = {
                'epoch': int(epoch),
                'train_loss': float(metrics.get('train_loss', 0.0)),
                'learning_rate': float(metrics.get('learning_rate', 0.0)),
                'epoch_time_sec': float(metrics.get('epoch_time_sec', metrics.get('epoch_time', 0.0))),
                'throughput': throughput,
                'gpu_memory_mb': gpu_memory_mb,
                'grad_norm': float(metrics.get('grad_norm', 0.0)),
                'skip_count': int(metrics.get('skip_count', 0))
            }
            self.epoch_metrics.append(row)
            
            fieldnames = list(row.keys())
            with open(self.logs_dir / 'epoch_metrics.csv', 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(row)
            
            # FORCE PRINT - EPOCH SUMMARY
            print(f" [EPOCH {epoch} SUMMARY] Loss: {row['train_loss']:.4f} | Time: {row['epoch_time_sec']:.1f}s", flush=True)
            
        except Exception as e:
            print(f"Error logging epoch: {e}")

    def log_validation(self, epoch: int, metrics: Dict[str, Any], val_time_sec: float):
        try:
            safe_row = {
                'epoch': int(epoch),
                'mAP': float(metrics.get('mAP', 0.0)),
                'AP50': float(metrics.get('AP50', 0.0)),
                'AP75': float(metrics.get('AP75', 0.0)),
                'AP_small': float(metrics.get('AP_small', 0.0)),
                'AP_medium': float(metrics.get('AP_medium', 0.0)),
                'AP_large': float(metrics.get('AP_large', 0.0)),
                'precision': float(metrics.get('precision', 0.0)),
                'recall': float(metrics.get('recall', 0.0)),
                'f1': float(metrics.get('f1', 0.0)),
                'val_time_sec': float(val_time_sec)
            }
            self.val_metrics.append(safe_row)
            
            fieldnames = list(safe_row.keys())
            with open(self.logs_dir / 'validation_metrics.csv', 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(safe_row)
            
            # --- FIX SYNTAX ERROR HERE: Split into multiple prints ---
            print(f"\n [VAL RESULT E{epoch}] mAP: {safe_row['mAP']:.4f} | AP50: {safe_row['AP50']:.4f} | F1: {safe_row['f1']:.4f}", flush=True)
            print(f"   Details: AP75={safe_row['AP75']:.3f}", flush=True)
            print(f"   Scales : Small={safe_row['AP_small']:.3f} | Medium={safe_row['AP_medium']:.3f} | Large={safe_row['AP_large']:.3f}\n", flush=True)
            
        except Exception as e:
            print(f"Error logging validation: {e}")

    # (Giữ nguyên hàm vẽ biểu đồ, nó sẽ dùng self.epoch_metrics đã load full history)
    def plot_training_progress(self):
        pass 

# Instantiate
metrics_logger = MetricsLoggerV2(CONFIG, logger)
logger.info("Metrics Logger V2 (Syntax Fixed) initialized")

In [ ]:
# ============================================================================
# CELL 4: MEMORY MONITOR (HARDENED, PRODUCTION)
# ============================================================================
from typing import Optional

class MemoryMonitor:
    """
    Tracks GPU memory usage during training/validation.
    Usage:
      - Call `reset()` at start of epoch (or when you want to zero peak).
      - Call `update()` after a forward/backward step to refresh current/peak stats.
      - Use `get_metrics()` to obtain a dict with 'current_mb' and 'peak_mb'.
    Notes:
      - Works safely on CPU-only environments (no-op).
      - `device` may be an int (cuda index) or torch.device; default uses current CUDA device 0.
    """
    def __init__(self, device: Optional[torch.device | int] = 0):
        self.peak_mb = 0.0
        self.current_mb = 0.0
        # store device index or None
        if isinstance(device, torch.device):
            if device.type == 'cuda':
                self.device = device.index
            else:
                self.device = None
        elif device is not None:
            self.device = int(device)
        else:
            self.device = None

    def _has_cuda(self) -> bool:
        return torch.cuda.is_available()

    def _device_index(self) -> Optional[int]:
        if not self._has_cuda():
            return None
        # If a device index was provided, use it; else default to current_device()
        try:
            return self.device if self.device is not None else torch.cuda.current_device()
        except Exception:
            return 0

    def update(self):
        """Update current and peak GPU memory usage safely."""
        if not self._has_cuda():
            # No GPU available — keep zeros
            return
        dev = self._device_index()
        try:
            # Prefer the 'reserved' (cached) memory as it reflects allocator usage
            if hasattr(torch.cuda, "memory_reserved"):
                reserved = torch.cuda.memory_reserved(dev)
            else:
                # fall back to allocated if reserved not available
                reserved = torch.cuda.memory_allocated(dev)
            # max memory reserved (peak) if available
            if hasattr(torch.cuda, "max_memory_reserved"):
                max_reserved = torch.cuda.max_memory_reserved(dev)
            elif hasattr(torch.cuda, "max_memory_allocated"):
                max_reserved = torch.cuda.max_memory_allocated(dev)
            else:
                # as last resort use current reserved/allocated as peak
                max_reserved = reserved
            self.current_mb = float(reserved) / (1024 ** 2)
            self.peak_mb = max(self.peak_mb, float(max_reserved) / (1024 ** 2))
        except Exception as e:
            # Log but do not raise to avoid breaking training loop
            logger.warning(f"MemoryMonitor.update() failed: {e}")

    def reset(self):
        """Reset peak and current stats. Also call CUDA reset_peak_memory_stats() if available."""
        self.peak_mb = 0.0
        self.current_mb = 0.0
        if not self._has_cuda():
            return
        dev = self._device_index()
        try:
            if hasattr(torch.cuda, "reset_peak_memory_stats"):
                torch.cuda.reset_peak_memory_stats(dev)
        except Exception as e:
            logger.warning(f"MemoryMonitor.reset() failed: {e}")

    def get_metrics(self) -> dict:
        """Return a small dict with current and peak memory in MB (floats)."""
        return {"current_mb": float(self.current_mb), "peak_mb": float(self.peak_mb)}

    def __repr__(self):
        return f"MemoryMonitor(current_mb={self.current_mb:.1f} MB, peak_mb={self.peak_mb:.1f} MB, device={self.device})"

# instantiate (default device 0 if CUDA available)
memory_monitor = MemoryMonitor(device=0 if torch.cuda.is_available() else None)
logger.info(f"MemoryMonitor initialized: {memory_monitor}")

In [ ]:
# ============================================================================
# CELL 5: SCIENTIFIC VISUALIZATIONS (HARDENED FOR HEADLESS SERVER)
# ============================================================================
import matplotlib
# --- FIX: Force Agg backend to prevent display errors on Kaggle ---
matplotlib.use('Agg') 
from matplotlib import pyplot as plt

# Try to set Times New Roman if available, otherwise leave default
_try_font = "Times New Roman"
try:
    available = matplotlib.font_manager.findfont(_try_font, fallback_to_default=False)
    matplotlib.rcParams['font.family'] = _try_font
except Exception:
    pass

# Global style tuned for publication
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.linewidth'] = 1.5
matplotlib.rcParams['xtick.major.width'] = 1.5
matplotlib.rcParams['ytick.major.width'] = 1.5
matplotlib.rcParams['lines.linewidth'] = 2.0
matplotlib.rcParams['lines.markersize'] = 6

class ScientificVisualizer:
    def __init__(self, output_dir, logger):
        self.output_dir = Path(output_dir)
        self.plots_dir = self.output_dir / 'plots' / 'figures'
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        self.logger = logger
        # color palette (ensure enough colors)
        self.colors = ['#0072B2', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#D55E00', '#2ECC71', '#E74C3C']

    # --- internal helpers -------------------------------------------------
    def _ensure_df_sorted(self, df, epoch_col='epoch'):
        if df is None or df.empty:
            return pd.DataFrame()
        df = df.copy()
        if epoch_col in df.columns:
            df = df.sort_values(epoch_col).reset_index(drop=True)
        return df

    def _safe_series(self, df, col, default=0.0):
        if df is None or df.empty or col not in df.columns:
            return np.array([default] * len(df)) if df is not None else np.array([])
        # convert to numpy and replace NaN with default
        arr = df[col].to_numpy(dtype=float)
        arr = np.nan_to_num(arr, nan=float(default), posinf=float(default), neginf=float(default))
        return arr

    # --- plotting methods -----------------------------------------------
    def plot_loss_convergence(self, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_loss_convergence")
            return None
        epoch_df = self._ensure_df_sorted(epoch_df)
        if epoch_df.empty:
            self.logger.warning("plot_loss_convergence: empty epoch_df")
            return None
        try:
            epochs = epoch_df['epoch'].to_numpy()
            loss = self._safe_series(epoch_df, 'train_loss', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, loss, color=self.colors[0], linewidth=2.5,
                    label='Training Loss', marker='o', markersize=4,
                    markevery=max(1, len(epochs)//10))
            ax.fill_between(epochs, loss, alpha=0.15, color=self.colors[0])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
            ax.set_title('Training Loss Convergence', fontsize=13, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='upper right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_01_loss_convergence.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_loss_convergence failed: {e}")
            return None

    def plot_learning_rate_schedule(self, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_learning_rate_schedule")
            return None
        epoch_df = self._ensure_df_sorted(epoch_df)
        if epoch_df.empty:
            self.logger.warning("plot_learning_rate_schedule: empty epoch_df")
            return None
        try:
            epochs = epoch_df['epoch'].to_numpy()
            lr = self._safe_series(epoch_df, 'learning_rate', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.semilogy(epochs, lr, color=self.colors[1], linewidth=2.5,
                        marker='s', markersize=4, markevery=max(1, len(epochs)//10), label='Learning Rate')
            ax.fill_between(epochs, lr, alpha=0.12, color=self.colors[1])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
            ax.set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, which='both')
            ax.legend(fontsize=11, loc='best', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_02_learning_rate.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_learning_rate_schedule failed: {e}")
            return None

    def plot_detection_metrics(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_detection_metrics")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_detection_metrics: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            mAP = self._safe_series(val_df, 'mAP', default=0.0)
            AP50 = self._safe_series(val_df, 'AP50', default=0.0)
            AP75 = self._safe_series(val_df, 'AP75', default=0.0)
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.plot(epochs, mAP, color=self.colors[0], linewidth=2.5, marker='o', markersize=5, label='mAP@IoU=0.50:0.95')
            ax.plot(epochs, AP50, color=self.colors[1], linewidth=2.0, marker='s', markersize=5, label='AP@IoU=0.50')
            ax.plot(epochs, AP75, color=self.colors[2], linewidth=2.0, marker='^', markersize=5, label='AP@IoU=0.75')
            ax.fill_between(epochs, mAP, alpha=0.1, color=self.colors[0])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Average Precision', fontsize=12, fontweight='bold')
            ax.set_title('Detection Performance at Different IoU Thresholds', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True, ncol=1)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_03_detection_metrics.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_detection_metrics failed: {e}")
            return None

    def plot_scale_analysis(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_scale_analysis")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_scale_analysis: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            AP_small = self._safe_series(val_df, 'AP_small', default=0.0)
            AP_medium = self._safe_series(val_df, 'AP_medium', default=0.0)
            AP_large = self._safe_series(val_df, 'AP_large', default=0.0)
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(epochs, AP_small, color=self.colors[3], linewidth=2.5, marker='o', markersize=5, label='Small Objects')
            ax.plot(epochs, AP_medium, color=self.colors[4], linewidth=2.5, marker='s', markersize=5, label='Medium Objects')
            ax.plot(epochs, AP_large, color=self.colors[5], linewidth=2.5, marker='^', markersize=5, label='Large Objects')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Average Precision', fontsize=12, fontweight='bold')
            ax.set_title('Scale-Dependent Performance', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_04_scale_analysis.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_scale_analysis failed: {e}")
            return None

    def plot_precision_curve(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_precision_curve")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_precision_curve: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            precision = self._safe_series(val_df, 'precision', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, precision, color=self.colors[6], linewidth=2.5, marker='o', markersize=5, label='Precision')
            ax.fill_between(epochs, precision, alpha=0.15, color=self.colors[6])
            if precision.size > 0:
                best_prec_idx = int(np.nanargmax(precision))
                ax.scatter(epochs[best_prec_idx], precision[best_prec_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {precision[best_prec_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
            ax.set_title('Precision Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_05_precision.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_precision_curve failed: {e}")
            return None

    def plot_recall_curve(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_recall_curve")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_recall_curve: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            recall = self._safe_series(val_df, 'recall', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, recall, color=self.colors[7], linewidth=2.5, marker='s', markersize=5, label='Recall')
            ax.fill_between(epochs, recall, alpha=0.15, color=self.colors[7])
            if recall.size > 0:
                best_recall_idx = int(np.nanargmax(recall))
                ax.scatter(epochs[best_recall_idx], recall[best_recall_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {recall[best_recall_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Recall', fontsize=12, fontweight='bold')
            ax.set_title('Recall Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_06_recall.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_recall_curve failed: {e}")
            return None

    def plot_f1_score(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_f1_score")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_f1_score: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            f1 = self._safe_series(val_df, 'f1', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, f1, color=self.colors[2], linewidth=2.5, marker='D', markersize=5, label='F1 Score')
            ax.fill_between(epochs, f1, alpha=0.15, color=self.colors[2])
            if f1.size > 0:
                best_f1_idx = int(np.nanargmax(f1))
                ax.scatter(epochs[best_f1_idx], f1[best_f1_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {f1[best_f1_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
            ax.set_title('F1 Score Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_07_f1_score.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_f1_score failed: {e}")
            return None

    def plot_map_per_epoch(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_map_per_epoch")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_map_per_epoch: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            maps = self._safe_series(val_df, 'mAP', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, maps, color=self.colors[5], linewidth=2.5, marker='o', markersize=5, label='mAP per epoch')
            ax.fill_between(epochs, maps, alpha=0.15, color=self.colors[5])
            if maps.size > 0:
                best_map_idx = int(np.nanargmax(maps))
                ax.scatter(epochs[best_map_idx], maps[best_map_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {maps[best_map_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('mAP', fontsize=12, fontweight='bold')
            ax.set_title('Validation mAP per Epoch', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_08_map_per_epoch.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_map_per_epoch failed: {e}")
            return None

    def create_results_table(self, val_df, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping create_results_table")
            return None
        val_df = self._ensure_df_sorted(val_df)
        epoch_df = self._ensure_df_sorted(epoch_df)
        if val_df.empty:
            self.logger.warning("create_results_table: empty val_df")
            return None
        try:
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.axis('tight')
            ax.axis('off')
            def get_best_info(df, col):
                if col not in df.columns or df.empty:
                    return ("N/A", "N/A", "N/A")
                best_idx = int(np.nanargmax(df[col].to_numpy(dtype=float)))
                best_val = float(df.iloc[best_idx][col])
                best_epoch = int(df.iloc[best_idx]['epoch']) if 'epoch' in df.columns else "N/A"
                final_val = float(df[col].iloc[-1]) if len(df) > 0 else "N/A"
                return (f"{best_val:.4f}", f"{best_epoch}", f"{final_val:.4f}")
            results_data = [
                ['Metric', 'Best Value', 'Epoch', 'Final Value'],
                ['mAP@50:95', *get_best_info(val_df, 'mAP')],
                ['AP@50', *get_best_info(val_df, 'AP50')],
                ['AP@75', *get_best_info(val_df, 'AP75')],
                ['F1 Score', *get_best_info(val_df, 'f1')],
            ]
            table = ax.table(cellText=results_data, cellLoc='center', loc='center', colWidths=[0.25]*4)
            table.auto_set_font_size(False)
            table.set_fontsize(11)
            table.scale(1, 2.0)
            # header styling
            for i in range(len(results_data[0])):
                cell = table[(0, i)]
                cell.set_facecolor('#0072B2')
                cell.get_text().set_color('white')
                cell.get_text().set_weight('bold')
            # zebra rows
            for i in range(1, len(results_data)):
                for j in range(len(results_data[0])):
                    if i % 2 == 0:
                        table[(i, j)].set_facecolor('#F0F0F0')
                    else:
                        table[(i, j)].set_facecolor('white')
            plt.title('Detection Results Summary', fontsize=13, fontweight='bold', pad=12)
            plt.tight_layout()
            savepath = self.plots_dir / 'table_01_results_summary.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Table saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"create_results_table failed: {e}")
            return None

    def generate_all_figures(self, epoch_df, val_df):
        self.logger.info("Generating publication-quality figures...")
        figures = []
        epoch_df = self._ensure_df_sorted(epoch_df)
        val_df = self._ensure_df_sorted(val_df)
        if not epoch_df.empty:
            figures.append(self.plot_loss_convergence(epoch_df))
            figures.append(self.plot_learning_rate_schedule(epoch_df))
        if not val_df.empty:
            figures.append(self.plot_detection_metrics(val_df))
            figures.append(self.plot_scale_analysis(val_df))
            figures.append(self.plot_precision_curve(val_df))
            figures.append(self.plot_recall_curve(val_df))
            figures.append(self.plot_f1_score(val_df))
            figures.append(self.plot_map_per_epoch(val_df))
        if not epoch_df.empty and not val_df.empty:
            figures.append(self.create_results_table(val_df, epoch_df))
        self.logger.info("All figures generated")
        self.logger.info(f"Output directory: {self.plots_dir}")
        return [f for f in figures if f is not None]

# convenience function
def generate_publication_figures(config, logger):
    visualizer = ScientificVisualizer(config['OUTPUT_DIR'], logger)
    epoch_csv = Path(config['OUTPUT_DIR']) / 'logs' / 'epoch_metrics.csv'
    val_csv = Path(config['OUTPUT_DIR']) / 'logs' / 'validation_metrics.csv'
    epoch_df = pd.read_csv(epoch_csv) if epoch_csv.exists() else pd.DataFrame()
    val_df = pd.read_csv(val_csv) if val_csv.exists() else pd.DataFrame()
    return visualizer.generate_all_figures(epoch_df, val_df)

logger.info("Scientific Visualizer loaded")

In [ ]:
# ============================================================================
# CELL 6: DATASET - FINAL (WITH SCALE INFO FOR VALIDATION)
# ============================================================================
from typing import Optional, List, Dict, Tuple
from pathlib import Path
import numpy as np
import torch
import cv2
import logging

logger = logging.getLogger(__name__)

try:
    cv2.setNumThreads(0)
except Exception:
    pass

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from pycocotools.coco import COCO
except Exception:
    pass

def get_transforms(train: bool = True, input_size: int = 448, include_bbox: bool = True):
    common_transforms = [
        A.LongestMaxSize(max_size=input_size),
        A.PadIfNeeded(min_height=input_size, min_width=input_size, border_mode=cv2.BORDER_CONSTANT), 
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]
    
    if include_bbox:
        if train:
            aug = [
                A.HorizontalFlip(p=0.5),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
                A.Blur(blur_limit=3, p=0.1),
                A.RandomBrightnessContrast(p=0.2),
            ]
            return A.Compose(aug + common_transforms, bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.1))
        return A.Compose(common_transforms, bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.1))
    else:
        return A.Compose(common_transforms)

def sanitize_bboxes_strict(bboxes: List[List[float]], img_shape: Tuple[int, int, int], min_size: float = 2.0) -> List[List[float]]:
    H, W = img_shape[0], img_shape[1]
    sanitized = []
    for bbox in bboxes:
        if len(bbox) < 4: continue
        x, y, w, h = float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])
        
        x1 = max(0.0, min(float(W - 1), x))
        y1 = max(0.0, min(float(H - 1), y))
        x2 = max(x1 + 1.0, min(float(W), x + w))
        y2 = max(y1 + 1.0, min(float(H), y + h))
        
        if (x2 - x1) < min_size or (y2 - y1) < min_size: continue
        sanitized.append([x1, y1, x2 - x1, y2 - y1]) # return scaled xywh temporarily for albumentations
    return sanitized

class MultiScaleTrafficDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir: str, ann_file: str, config: Dict, train: bool = True):
        super().__init__()
        self.img_dir = Path(img_dir)
        self.coco = COCO(str(ann_file))
        self.config = config
        self.train = train
        self.current_size = config['BASE_INPUT_SIZE']
        self.min_box_size = config.get('MIN_BOX_SIZE', 2.0) 
        
        self.cat_ids = sorted(self.coco.getCatIds())
        self.catid2label = {cid: i+1 for i, cid in enumerate(self.cat_ids)}
        
        self.valid_img_ids = []
        all_ids = self.coco.getImgIds()
        for img_id in all_ids:
            anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
            if len(anns) > 0:
                self.valid_img_ids.append(img_id)
        
        self.error_count = 0 
        logger.info(f"Dataset loaded: {len(self.valid_img_ids)} valid images.")

    def __len__(self):
        return len(self.valid_img_ids)

    def _load_safe_image(self, img_path):
        try:
            image = cv2.imread(img_path)
            if image is None: raise ValueError("Image None")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            return image
        except Exception:
            return np.zeros((self.current_size, self.current_size, 3), dtype=np.uint8)

    def __getitem__(self, idx):
        cv2.setNumThreads(0)
        
        img_id = self.valid_img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = str(self.img_dir / img_info['file_name'])
        
        image = self._load_safe_image(img_path)
        orig_h, orig_w = image.shape[:2] # <--- Lưu kích thước gốc

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        raw_bboxes, raw_labels = [], []
        
        for ann in anns:
            if ann['category_id'] not in self.catid2label: continue
            bbox = ann.get('bbox')
            if not bbox: continue
            raw_bboxes.append(bbox)
            raw_labels.append(self.catid2label[ann['category_id']])

        try:
            # Sanitize boxes before transform
            # Albumentations expects [x, y, w, h] for 'coco' format
            transform = get_transforms(self.train, self.current_size, include_bbox=True)
            transformed = transform(image=image, bboxes=raw_bboxes, labels=raw_labels)
            image_t = transformed['image']
            bboxes_t = transformed['bboxes']
            labels_t = transformed['labels']

        except Exception as e:
            if self.error_count < 5: 
                logger.warning(f"Transform fail {img_id}: {e}. Fallback.")
            self.error_count += 1
            # Fallback pipeline
            image_resized = cv2.resize(image, (self.current_size, self.current_size))
            img_float = image_resized.astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_norm = (img_float - mean) / std
            image_t = torch.from_numpy(img_norm.transpose(2, 0, 1)).float()
            bboxes_t, labels_t = [], []

        # Final format for Model: XYXY
        valid_boxes, valid_labels = [], []
        for i, box in enumerate(bboxes_t):
            x, y, w, h = box
            # Clamp & Convert to XYXY
            x1 = max(0, min(self.current_size, x))
            y1 = max(0, min(self.current_size, y))
            x2 = max(x1, min(self.current_size, x + w))
            y2 = max(y1, min(self.current_size, y + h))
            
            if (x2 - x1) > 1.0 and (y2 - y1) > 1.0: 
                 valid_boxes.append([x1, y1, x2, y2]) 
                 valid_labels.append(labels_t[i])

        if len(valid_boxes) > 0:
            boxes_final = torch.tensor(valid_boxes, dtype=torch.float32)
            labels_final = torch.tensor(valid_labels, dtype=torch.int64)
        else:
            boxes_final = torch.zeros((0, 4), dtype=torch.float32)
            labels_final = torch.zeros((0,), dtype=torch.int64)

        target = {
            'boxes': boxes_final,
            'labels': labels_final,
            'image_id': torch.tensor([img_id]),
            'orig_size': torch.tensor([orig_h, orig_w]), # <--- QUAN TRỌNG: Gửi size gốc cho Validation
            'img_size': torch.tensor([self.current_size, self.current_size]) # Size hiện tại
        }
        
        return image_t, target
        
    def set_input_size(self, size):
        self.current_size = size

logger.info("MultiScaleTrafficDataset loaded (With Scale Info)")

In [ ]:
# ============================================================================
# CELL 7: MODEL BUILDING (FREEZE BATCHNORM - NO MORE INF)
# ============================================================================

import math
import torch
import torch.nn as nn
from torchvision.models.detection.anchor_utils import AnchorGenerator

# Defensive references
if retinanet_resnet50_fpn_v2 is None:
    raise RuntimeError("retinanet_resnet50_fpn_v2 is not available.")

def freeze_batchnorm_layers(model):
    """
    Đóng băng các lớp BatchNorm. Đây là kỹ thuật chuẩn khi fine-tune object detection
    để tránh nổ gradient khi batch size nhỏ hoặc data nhiễu.
    """
    frozen_count = 0
    for module in model.modules():
        if isinstance(module, torch.nn.BatchNorm2d):
            module.eval() # Chuyển sang chế độ inference (dùng running mean/var cố định)
            for param in module.parameters():
                param.requires_grad = False # Không cập nhật trọng số BN
            frozen_count += 1
    logger.info(f"MODEL: Froze {frozen_count} BatchNorm layers for stability.")

def build_model(num_classes: int, device: torch.device, prior_prob: float = 0.01):
    logger.info("MODEL: Loading RetinaNet ResNet50 FPN v2...")
    if RetinaNet_ResNet50_FPN_V2_Weights is not None:
        model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
    else:
        model = retinanet_resnet50_fpn_v2(weights=None)

    # --- ANCHOR TUNING ---
    anchor_sizes = ((16, 24, 32), (32, 48, 64), (64, 96, 128), (128, 192, 256), (256, 320, 400))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    new_anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    model.anchor_generator = new_anchor_generator
    num_anchors = new_anchor_generator.num_anchors_per_location()[0]
    
    # --- HEAD REPLACEMENT ---
    in_channels = 256 
    new_cls_head = RetinaNetClassificationHead(
        in_channels=in_channels, 
        num_anchors=num_anchors, 
        num_classes=num_classes
    )
    model.head.classification_head = new_cls_head

    # --- BIAS INIT ---
    try:
        prior_bias = -math.log((1.0 - prior_prob) / prior_prob)
        for name, param in model.head.classification_head.named_parameters():
            if "cls_logits" in name and "bias" in name:
                torch.nn.init.constant_(param, prior_bias)
    except Exception:
        pass

    # --- CRITICAL FIX: FREEZE BATCHNORM ---
    # Bắt buộc thực hiện để tránh gradient explosion
    freeze_batchnorm_layers(model)

    model = model.to(device)
    return model

logger.info("Model builder loaded (BatchNorm Frozen)")

In [ ]:
# ============================================================================
# CELL 8: VALIDATION FUNCTION (FIXED PRECISION & RECALL LOGGING)
# ============================================================================

import time
import json
import numpy as np
import torch
from tqdm.notebook import tqdm
import io
from contextlib import redirect_stdout

def validate_model(
    model, val_loader, device, config: Dict, logger,
    category_id_map: Optional[Dict[int, int]] = None,
    save_predictions_path: Optional[str] = None
):
    model.eval()
    print("\n--- Starting Validation (Rescale Mode) ---", flush=True)
    start_time = time.time()

    try:
        coco_gt = COCO(str(config['VAL_ANN']))
    except Exception as e:
        print(f"Error loading COCO: {e}", flush=True)
        return {'mAP': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'inference_time': 0.0}

    all_predictions = []
    score_thresh = 0.01

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            images, targets = batch
            if images is None or targets is None: continue
            
            if isinstance(images, (list, tuple)): images = [img.to(device) for img in images]
            else: images = images.to(device)

            try:
                outputs = model(images)
            except: continue

            for i, out in enumerate(outputs):
                tgt = targets[i]
                img_id = int(tgt.get('image_id').item()) if isinstance(tgt, dict) and 'image_id' in tgt else 0
                
                # --- LOGIC RESCALE MỚI ---
                if 'orig_size' in tgt and 'img_size' in tgt:
                    orig_h, orig_w = tgt['orig_size'].tolist()
                    curr_h, curr_w = tgt['img_size'].tolist()
                    scale_x = orig_w / curr_w
                    scale_y = orig_h / curr_h
                else:
                    scale_x, scale_y = 1.0, 1.0

                boxes = out['boxes'].cpu().numpy()
                scores = out['scores'].cpu().numpy()
                labels = out['labels'].cpu().numpy()

                keep = scores > score_thresh
                boxes = boxes[keep]
                scores = scores[keep]
                labels = labels[keep]

                if len(boxes) == 0: continue

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
                    
                    # Rescale
                    x1 *= scale_x
                    y1 *= scale_y
                    x2 *= scale_x
                    y2 *= scale_y
                    
                    w, h = x2 - x1, y2 - y1
                    if w <= 0 or h <= 0: continue
                    
                    cat_id = int(label)
                    if category_id_map: cat_id = category_id_map.get(cat_id, cat_id)

                    all_predictions.append({
                        'image_id': img_id,
                        'category_id': cat_id,
                        'bbox': [x1, y1, w, h],
                        'score': float(score)
                    })

    inference_time = time.time() - start_time
    
    # --- KHỞI TẠO ĐẦY ĐỦ KEYS ---
    metrics = {
        'mAP': 0.0, 'AP50': 0.0, 'AP75': 0.0, 
        'AP_small': 0.0, 'AP_medium': 0.0, 'AP_large': 0.0,
        'f1': 0.0, 'precision': 0.0, 'recall': 0.0, # <--- Đã thêm key này
        'inference_time': inference_time
    }

    if not all_predictions:
        print("⚠️ WARNING: No predictions above threshold.", flush=True)
        return metrics

    try:
        with redirect_stdout(io.StringIO()):
            coco_dt = coco_gt.loadRes(all_predictions)
            coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()

        stats = coco_eval.stats
        # Map các chỉ số COCO sang metrics
        metrics['mAP'] = float(stats[0])       # AP @ IoU=0.50:0.95
        metrics['AP50'] = float(stats[1])      # AP @ IoU=0.50 (Dùng làm Precision)
        metrics['AP75'] = float(stats[2])
        metrics['AP_small'] = float(stats[3])
        metrics['AP_medium'] = float(stats[4])
        metrics['AP_large'] = float(stats[5])
        
        # --- FIX: Gán giá trị Precision & Recall ---
        # Trong object detection, AP50 thường được coi là Precision tổng quát
        metrics['precision'] = float(stats[1]) 
        # stats[8] là Average Recall @ IoU=0.50:0.95 | maxDets=100
        metrics['recall'] = float(stats[8])    

        if metrics['mAP'] > 0:
             metrics['f1'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'] + 1e-6)
        
        print(f"✅ Evaluated {len(all_predictions)} preds. mAP={metrics['mAP']:.4f} | Prec={metrics['precision']:.4f} | Rec={metrics['recall']:.4f}", flush=True)

    except Exception as e:
        print(f"COCO Eval Error: {e}", flush=True)

    return metrics

In [ ]:
# ============================================================================
# CELL 9: TRAINING LOOP (LOSS GUARD & SAFETY FUSE V2)
# ============================================================================
import gc
import time
from contextlib import nullcontext
from torch.cuda.amp import autocast, GradScaler

def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    epoch: int,
    scaler: Optional[GradScaler],
    scheduler=None,
    config: dict = {},
    metrics_logger=None
):
    """
    Train model for a single epoch with Multi-Layer Safety Fuse.
    """
    logger.info(f"[EPOCH {epoch}] Starting training loop...")
    
    model.train()
    mem_monitor = MemoryMonitor()
    mem_monitor.reset()

    losses = []
    grad_norms = []
    skip_count = 0
    global_update_steps = 0

    # Setup Multi-scale training
    if config.get('MULTI_SCALE', False):
        input_sizes = config.get('INPUT_SIZES', [config.get('BASE_INPUT_SIZE', 448)])
        scale_freq = config.get('SCALE_CHANGE_FREQUENCY', 100)
        current_size = random.choice(input_sizes)
        if hasattr(dataloader.dataset, 'set_input_size'):
            dataloader.dataset.set_input_size(current_size)
        logger.info(f"[EPOCH {epoch}] Starting with input size: {current_size}px")

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}", leave=True)
    start_time = time.time()
    optimizer.zero_grad()

    accumulation_steps = int(config.get('ACCUMULATION_STEPS', 1))
    log_interval = int(config.get('LOG_BATCH_INTERVAL', 50))
    grad_clip_value = float(config.get('GRAD_CLIP', 0.1)) # Default to strict clipping
    use_amp = config['USE_AMP']
    amp_ctx = autocast(enabled=use_amp) if use_amp else nullcontext()
    
    logger.info(f"[EPOCH {epoch}] DataLoader ready. Iterating batches...")

    for step, batch in enumerate(pbar, 1):
        # Multi-scale adjustment
        if config.get('MULTI_SCALE', False) and (step % scale_freq == 0):
            new_size = random.choice(input_sizes)
            if hasattr(dataloader.dataset, 'set_input_size'):
                dataloader.dataset.set_input_size(new_size)
                current_size = new_size

        if batch is None: continue
        try:
            images, targets = batch
        except Exception: continue
        if images is None or targets is None: continue

        # Move to device safely
        try:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        except Exception as e:
            logger.error(f"Device move error: {e}")
            continue

        # --- FORWARD PASS ---
        try:
            with amp_ctx:
                loss_dict = model(images, targets)
                loss = sum(loss for loss in loss_dict.values())
                
                # --- LAYER 1: CHECK LOSS ---
                if not torch.isfinite(loss):
                    print(f"\n[CRITICAL WARN] Epoch {epoch} Step {step}: Loss is {loss.item()} (Infinite/NaN). Skipping batch.", flush=True)
                    for k, v in loss_dict.items():
                        print(f"  - {k}: {v.item()}", flush=True)
                    optimizer.zero_grad()
                    skip_count += 1
                    continue

                loss = loss / accumulation_steps
        except Exception as e:
            logger.error(f"Forward error at step {step}: {e}")
            optimizer.zero_grad()
            continue

        # --- BACKWARD PASS ---
        try:
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
        except Exception as e:
            logger.error(f"Backward error at step {step}: {e}")
            optimizer.zero_grad()
            continue

        # --- OPTIMIZER STEP ---
        if (step % accumulation_steps) == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)

            # --- LAYER 2: CHECK GRADIENT NORM ---
            # Clip gradient để tránh nổ trọng số
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_value)
            
            if not torch.isfinite(grad_norm):
                print(f"\n[WARN] Epoch {epoch} Step {step}: GradNorm is {grad_norm} (Infinite). Skipping update.", flush=True)
                optimizer.zero_grad()
                skip_count += 1
                if scaler is not None: scaler.update()
                continue

            # Cảnh báo nếu gradient quá lớn dù đã clip (dấu hiệu bất ổn)
            if grad_norm > 5.0:
                pass # Có thể uncomment dòng dưới để debug sâu
                # print(f"[INFO] High GradNorm: {grad_norm:.2f}", flush=True)

            # Step
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            if scheduler is not None:
                scheduler.step()

            optimizer.zero_grad()
            global_update_steps += 1
            grad_norms.append(float(grad_norm))

        # Logging & Monitoring
        try:
            current_loss = loss.item() * accumulation_steps
            losses.append(current_loss)
        except: pass
        
        mem_monitor.update()

        if step % log_interval == 0 or step == len(dataloader):
            avg_loss_recent = float(np.mean(losses[-log_interval:])) if losses else 0.0
            current_lr = optimizer.param_groups[0].get('lr', 0.0)
            
            # Update Progress Bar
            pbar.set_postfix({
                'loss': f'{avg_loss_recent:.4f}',
                'lr': f'{current_lr:.2e}',
                'mem': f'{mem_monitor.current_mb:.0f}MB'
            })
            
            # --- FORCE FLUSH LOGGING ---
            # In thẳng ra stdout để hiện trên Kaggle Log
            print(f"[E{epoch}|{step}/{len(dataloader)}] Loss: {avg_loss_recent:.4f} | LR: {current_lr:.2e} | Mem: {mem_monitor.current_mb:.0f}MB", flush=True)

    epoch_time = time.time() - start_time
    avg_loss = float(np.mean(losses)) if losses else 0.0
    avg_grad_norm = float(np.mean([g for g in grad_norms if np.isfinite(g)])) if grad_norms else 0.0
    avg_throughput = (len(dataloader.dataset) / epoch_time) if epoch_time > 0 else 0.0

    metrics = {
        'train_loss': avg_loss,
        'learning_rate': optimizer.param_groups[0].get('lr', 0.0),
        'epoch_time_sec': epoch_time,
        'throughput': avg_throughput,
        'gpu_memory_mb': mem_monitor.peak_mb,
        'grad_norm': avg_grad_norm,
        'skip_count': skip_count,
        'update_steps': global_update_steps
    }

    if metrics_logger:
        try:
            metrics_logger.log_epoch(epoch, metrics)
        except Exception: pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    logger.info(f"[EPOCH {epoch}] Finished. Avg Loss: {avg_loss:.4f}. Skipped Batches: {skip_count}")

    
    return metrics

logger.info("Training loop loaded (Loss Guard V2 + Force Flush)")

In [ ]:
# ============================================================================
# CELL 10: TEST EVALUATOR (FIXED RESCALE & LABEL MAPPING)
# ============================================================================
import os
import time
import json
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional
import numpy as np
import torch
import cv2
from PIL import Image
from tqdm.notebook import tqdm

# Force matplotlib backend to avoid display errors
import matplotlib
try:
    matplotlib.use('Agg')
except:
    pass

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    COCO = None
    COCOeval = None

class ComprehensiveEvaluator:
    """Complete test evaluation with proper metrics and rescaling."""
    
    def __init__(self, test_img_dir, test_ann_file, logger, config):
        self.test_img_dir = test_img_dir
        self.test_ann_file = test_ann_file
        self.logger = logger
        self.config = config
        self.output_dir = Path(config['OUTPUT_DIR'])
        self.plots_dir = self.output_dir / 'plots'
        self.logs_dir = self.output_dir / 'logs'
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        self.logs_dir.mkdir(parents=True, exist_ok=True)
        
        try:
            self.coco_test = COCO(str(test_ann_file))
            self.test_ids = self.coco_test.getImgIds()
            self.logger.info(f"TEST: Loaded {len(self.test_ids)} test images")
        except Exception as e:
            self.logger.error(f"TEST: Failed to load COCO: {e}")
            self.coco_test = None
            self.test_ids = []
    
    def evaluate_on_test_set(self, model, device, classnames, iou_threshold=0.5, score_threshold=0.01, category_mapping=None):
        """
        Run complete evaluation.
        Args:
            category_mapping: Dict mapping model_label_index -> coco_category_id
                              (Nếu None, mặc định giữ nguyên index)
        """
        
        if not self.coco_test:
            self.logger.warning("TEST: COCO not available")
            return None
        
        model.eval()
        model = model.to(device)
        all_predictions = []
        all_ground_truth = []
        inference_times = []
        
        self.logger.info("=" * 80)
        self.logger.info("TEST: Starting evaluation (Rescale Mode)...")
        
        # Lấy target size từ config
        target_size = self.config.get('BASE_INPUT_SIZE', 432)
        
        with torch.no_grad():
            pbar = tqdm(self.test_ids, desc="Test Eval", leave=False)
            
            for img_id in pbar:
                img_info_list = self.coco_test.loadImgs(img_id)
                if not img_info_list: continue
                
                img_info = img_info_list[0]
                img_path = os.path.join(self.test_img_dir, img_info['file_name'])
                
                if not os.path.exists(img_path): continue

                # --- 1. Load Image & Get Original Size ---
                try:
                    # Dùng PIL để lấy kích thước gốc nhanh và chính xác
                    image_pil = Image.open(img_path).convert('RGB')
                    orig_w, orig_h = image_pil.size
                    image_np = np.array(image_pil)
                except Exception as e:
                    self.logger.warning(f"TEST: Failed to load {img_path}: {e}")
                    continue
                
                # --- 2. Prepare Input (Resize & Normalize) ---
                image_tensor = self._prepare_input(image_np, device, target_size)
                
                t0 = time.time()
                try:
                    outputs = model([image_tensor])
                except Exception as e:
                    self.logger.error(f"Inference failed for {img_id}: {e}")
                    continue
                
                inference_time = (time.time() - t0) * 1000
                inference_times.append(inference_time)
                
                output = outputs[0]
                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                
                # Tính tỷ lệ Scale: Gốc / Model_Input
                # Model input là vuông (target_size x target_size)
                scale_x = orig_w / target_size
                scale_y = orig_h / target_size
                
                # --- 3. Process Predictions ---
                if len(scores) > 100: # Limit top 100 boxes
                    top_k = np.argsort(scores)[-100:]
                    boxes = boxes[top_k]
                    scores = scores[top_k]
                    labels = labels[top_k]
                
                for box, score, label in zip(boxes, scores, labels):
                    if score > score_threshold:
                        # Lấy toạ độ trên ảnh input (432)
                        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
                        
                        # --- FIX QUAN TRỌNG: Rescale về ảnh gốc ---
                        x1 *= scale_x
                        y1 *= scale_y
                        x2 *= scale_x
                        y2 *= scale_y
                        
                        w, h = x2 - x1, y2 - y1
                        if w <= 0 or h <= 0: continue

                        # Map Model Label Index -> COCO Category ID
                        cat_id = int(label)
                        if category_mapping:
                            cat_id = category_mapping.get(cat_id, cat_id)

                        all_predictions.append({
                            'image_id': int(img_id),
                            'category_id': cat_id,
                            'bbox': [x1, y1, w, h], # COCO yêu cầu XYWH
                            'score': float(score)
                        })
                
                # Collect GT for F1 calculation
                anns = self.coco_test.loadAnns(self.coco_test.getAnnIds(imgIds=img_id))
                for ann in anns:
                    x, y, w, h = ann['bbox']
                    all_ground_truth.append({
                        'image_id': int(img_id),
                        'category_id': ann['category_id'],
                        'bbox': [float(x), float(y), float(w), float(h)]
                    })
        
        # --- COCO Metrics ---
        coco_metrics = {k: 0.0 for k in ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']}
        if all_predictions:
            try:
                import io
                from contextlib import redirect_stdout
                
                with redirect_stdout(io.StringIO()):
                    coco_dt = self.coco_test.loadRes(all_predictions)
                    coco_eval = COCOeval(self.coco_test, coco_dt, 'bbox')
                    coco_eval.evaluate()
                    coco_eval.accumulate()
                    coco_eval.summarize()
                
                stats = coco_eval.stats
                coco_metrics['mAP'] = float(stats[0])
                coco_metrics['AP50'] = float(stats[1])
                coco_metrics['AP75'] = float(stats[2])
                coco_metrics['AP_small'] = float(stats[3])
                coco_metrics['AP_medium'] = float(stats[4])
                coco_metrics['AP_large'] = float(stats[5])
            except Exception as e:
                self.logger.error(f"TEST: COCO eval failed: {e}")
        else:
            self.logger.warning("TEST: No predictions generated.")
        
        # --- F1 Metrics ---
        f1_metrics = self._compute_f1_metrics(all_predictions, all_ground_truth, classnames, iou_threshold)
        
        # --- Inference Stats ---
        inference_stats = {
            'avg_ms': float(np.mean(inference_times)) if inference_times else 0,
            'fps': 1000.0 / np.mean(inference_times) if inference_times and np.mean(inference_times) > 0 else 0,
            'total_time_sec': float(sum(inference_times) / 1000) if inference_times else 0
        }
        
        self._log_test_results({'coco_metrics': coco_metrics, 'f1_metrics': f1_metrics, 'inference_stats': inference_stats}, classnames)
        return {'coco_metrics': coco_metrics, 'f1_metrics': f1_metrics}
    
    def _prepare_input(self, image_np, device, target_size):
        """Prepare normalized tensor with resize."""
        try: cv2.setNumThreads(0)
        except: pass

        MEAN = [0.485, 0.456, 0.406]
        STD = [0.229, 0.224, 0.225]
        
        try:
            image_resized = cv2.resize(image_np, (target_size, target_size))
        except Exception:
            image_resized = np.array(Image.fromarray(image_np).resize((target_size, target_size)))

        image_norm = image_resized.astype(np.float32) / 255.0
        image_norm = (image_norm - np.array(MEAN)) / np.array(STD)
        
        return torch.from_numpy(image_norm.transpose(2, 0, 1)).float().to(device)
    
    @staticmethod
    def _compute_iou(box1_xyxy, box2_xyxy):
        box1 = np.array(box1_xyxy, dtype=np.float32)
        box2 = np.array(box2_xyxy, dtype=np.float32)
        
        x1_min = max(box1[0], box2[0])
        y1_min = max(box1[1], box2[1])
        x2_max = min(box1[2], box2[2])
        y2_max = min(box1[3], box2[3])
        
        if x2_max <= x1_min or y2_max <= y1_min: return 0.0
        
        inter_area = (x2_max - x1_min) * (y2_max - y1_min)
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union_area = box1_area + box2_area - inter_area
        
        return inter_area / union_area if union_area > 0 else 0.0
    
    def _compute_f1_metrics(self, predictions, ground_truths, classnames, iou_threshold=0.5):
        gt_by_class = defaultdict(list)
        pred_by_class = defaultdict(list)
        
        for gt in ground_truths: gt_by_class[gt['category_id']].append(gt['bbox'])
        for pred in predictions: pred_by_class[pred['category_id']].append({'bbox': pred['bbox'], 'score': pred['score']})
        
        class_metrics = {}
        for class_id in range(1, len(classnames) + 1):
            gt_boxes = gt_by_class.get(class_id, [])
            preds = sorted(pred_by_class.get(class_id, []), key=lambda x: x['score'], reverse=True)
            
            tp, fp = 0, 0
            matched_gt = set()
            
            for pred in preds:
                px, py, pw, ph = pred['bbox']
                pred_xyxy = [px, py, px + pw, py + ph]
                
                best_iou, best_gt_idx = 0.0, -1
                for gt_idx, (gx, gy, gw, gh) in enumerate(gt_boxes):
                    if gt_idx in matched_gt: continue
                    iou = self._compute_iou(pred_xyxy, [gx, gy, gx + gw, gy + gh])
                    if iou > best_iou: best_iou, best_gt_idx = iou, gt_idx
                
                if best_iou >= iou_threshold:
                    tp += 1
                    matched_gt.add(best_gt_idx)
                else:
                    fp += 1
            
            fn = len(gt_boxes) - len(matched_gt)
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            
            cname = classnames[class_id - 1] if (class_id - 1) < len(classnames) else str(class_id)
            class_metrics[cname] = {'precision': precision, 'recall': recall, 'f1': f1}
            
        if class_metrics:
            macro = {
                'precision': float(np.mean([m['precision'] for m in class_metrics.values()])),
                'recall': float(np.mean([m['recall'] for m in class_metrics.values()])),
                'f1': float(np.mean([m['f1'] for m in class_metrics.values()]))
            }
        else: macro = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        return {'per_class': class_metrics, 'macro': macro}
    
    def _log_test_results(self, test_results, classnames):
        coco = test_results['coco_metrics']
        f1m = test_results['f1_metrics']
        self.logger.info(f"TEST RESULTS: mAP={coco['mAP']:.4f} | AP50={coco['AP50']:.4f} | FPS={test_results['inference_stats']['fps']:.1f}")
        for c, m in f1m['per_class'].items():
            self.logger.info(f"  {c}: F1={m['f1']:.4f} (P={m['precision']:.2f}, R={m['recall']:.2f})")

logger.info("Test Evaluator loaded (Rescale Fixed)")

In [ ]:
# ============================================================================
# CELL 11: EXPORT UTILITIES (ROBUST WRAPPER & CPU FALLBACK)
# ============================================================================
import torch
import torch.nn as nn
from pathlib import Path

# --- 1. WRAPPER CLASS (Quan trọng để fix lỗi output format) ---
class DetectorExportWrapper(nn.Module):
    """
    Wrapper giúp model RetinaNet trả về Tensor phẳng (Tuple) thay vì List[Dict].
    Bắt buộc phải có để export TorchScript và ONNX thành công.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # Gọi model gốc
        # RetinaNet trả về: [{'boxes': ..., 'scores': ..., 'labels': ...}, ...]
        predictions = self.model(x)
        
        # Lấy kết quả của ảnh đầu tiên (batch_size=1 khi export)
        pred = predictions[0]
        
        # Trả về Tuple(Tensor, Tensor, Tensor) -> Dễ chịu cho TorchScript/ONNX
        return pred['boxes'], pred['scores'], pred['labels']

# --- 2. EXPORTER CLASS ---
class ModelExporter:
    """Export model to various formats safely."""
    
    def __init__(self, output_dir, logger):
        self.output_dir = Path(output_dir)
        self.logger = logger
        self.export_dir = self.output_dir / 'exports'
        self.export_dir.mkdir(exist_ok=True)
    
    def export_torchscript(self, model, input_size):
        """Export to TorchScript format using Wrapper and CPU."""
        try:
            savepath = self.export_dir / 'model_traced.pt'
            
            # Chuyển sang CPU để export an toàn (tránh OOM)
            device_cpu = torch.device('cpu')
            model_cpu = model.to(device_cpu)
            model_cpu.eval()
            
            # Wrap model
            wrapped_model = DetectorExportWrapper(model_cpu)
            wrapped_model.eval()
            
            # Dummy input on CPU
            dummy_input = torch.randn(1, 3, input_size, input_size).to(device_cpu)
            
            with torch.no_grad():
                traced_model = torch.jit.trace(wrapped_model, dummy_input)
            
            traced_model.save(str(savepath))
            
            file_size_mb = savepath.stat().st_size / (1024 * 1024)
            self.logger.info(f"EXPORT: TorchScript (Traced) saved: {savepath} ({file_size_mb:.1f}MB)")
            
            return savepath
        
        except Exception as e:
            self.logger.error(f"EXPORT: TorchScript export failed: {e}")
            return None
    
    def export_onnx(self, model, input_size, opset_version=12):
        """Export to ONNX (best for inference) using Wrapper and CPU."""
        try:
            import onnx
            
            output_path = self.export_dir / 'model.onnx'
            
            # Chuyển sang CPU
            device_cpu = torch.device('cpu')
            model_cpu = model.to(device_cpu)
            model_cpu.eval()
            
            # Wrap model
            wrapped_model = DetectorExportWrapper(model_cpu)
            wrapped_model.eval()
            
            dummy_input = torch.randn(1, 3, input_size, input_size).to(device_cpu)
            
            with torch.no_grad():
                torch.onnx.export(
                    wrapped_model,
                    dummy_input,
                    str(output_path),
                    opset_version=opset_version,
                    do_constant_folding=True,
                    input_names=['input'],
                    output_names=['boxes', 'scores', 'labels'],
                    dynamic_axes={
                        'input': {0: 'batch'}, 
                        'boxes': {0: 'num_dets'}, 
                        'scores': {0: 'num_dets'}, 
                        'labels': {0: 'num_dets'}
                    },
                    verbose=False
                )
            
            file_size_mb = output_path.stat().st_size / (1024 * 1024)
            self.logger.info(f"EXPORT: ONNX saved: {output_path} ({file_size_mb:.1f}MB)")
            
            # Verify ONNX
            try:
                onnx_model = onnx.load(str(output_path))
                onnx.checker.check_model(onnx_model)
                self.logger.info("EXPORT: ONNX verification passed")
            except Exception as e:
                self.logger.warning(f"EXPORT: ONNX verification failed: {e}")
            
            return output_path
        
        except ImportError:
            self.logger.warning("EXPORT: ONNX not installed, skipping")
            return None
        except Exception as e:
            self.logger.error(f"EXPORT: ONNX export failed: {e}")
            return None

    def export_checkpoint(self, model, optimizer, scheduler, epoch, metrics, config):
        """Export training checkpoint (Full State)."""
        try:
            savepath = self.export_dir / f'checkpoint_epoch_{epoch}.pth'
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'config': config, # Lưu config để tiện resume
                'metrics': metrics
            }
            
            torch.save(checkpoint, str(savepath))
            file_size_mb = savepath.stat().st_size / (1024 * 1024)
            self.logger.info(f"EXPORT: Checkpoint saved: {savepath} ({file_size_mb:.1f}MB)")
            return savepath
        except Exception as e:
            self.logger.error(f"EXPORT: Checkpoint export failed: {e}")
            return None
            
    def export_state_dict(self, model):
        """Export weights only."""
        try:
            savepath = self.export_dir / 'model_weights_only.pth'
            torch.save(model.state_dict(), str(savepath))
            file_size_mb = savepath.stat().st_size / (1024 * 1024)
            self.logger.info(f"EXPORT: Weights only saved: {savepath} ({file_size_mb:.1f}MB)")
            return savepath
        except Exception as e:
            self.logger.error(f"EXPORT: Weights export failed: {e}")
            return None

# --- MAIN FUNCTION TO CALL ---
def run_exports(model, device, config, logger, optimizer=None, scheduler=None, epoch=None):
    """Run all exports safely."""
    if not config.get('EXPORT_MODEL', False):
        return {}

    logger.info("-" * 40)
    logger.info("EXPORT: Starting model exports...")

    # Clear Cache GPU trước khi export
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    exporter = ModelExporter(config['OUTPUT_DIR'], logger)
    input_size = config.get('BASE_INPUT_SIZE', 432)
    
    exports = {}

    # 1. Export TorchScript (Priority)
    # Lưu ý: Truyền model gốc vào, hàm export sẽ tự wrap
    exports['torchscript'] = exporter.export_torchscript(model, input_size)

    # 2. Export ONNX
    exports['onnx'] = exporter.export_onnx(model, input_size)

    # 3. Export Weights
    exports['weights'] = exporter.export_state_dict(model)

    # 4. Export Full Checkpoint (Optional)
    if optimizer and epoch is not None:
        exports['checkpoint'] = exporter.export_checkpoint(model, optimizer, scheduler, epoch, {}, config)

    # Đưa model trở lại thiết bị gốc (GPU) để tiếp tục train (nếu cần)
    model.to(device)
    
    logger.info(f"EXPORT: Completed. Dir: {exporter.export_dir}")
    logger.info("-" * 40)
    return exports

logger.info("Export utilities loaded (Robust V2 with Wrapper)")

In [ ]:
# ============================================================================
# CELL 12: MAIN TRAINING PIPELINE (ADAMW + RESUME + AUTO TEST)
# ============================================================================
from torch.utils.data import Subset
import os

def collate_fn(batch):
    # Filter failed images (None) from dataset
    batch = [item for item in batch if item is not None]
    if len(batch) == 0: return None
    return [item[0] for item in batch], [item[1] for item in batch]

def main():
    logger.info("=" * 80); logger.info("MAIN PIPELINE START (ADAMW)"); logger.info("=" * 80)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Datasets
    train_dataset = MultiScaleTrafficDataset(CONFIG['TRAIN_IMG'], CONFIG['TRAIN_ANN'], CONFIG, train=True)
    val_dataset = MultiScaleTrafficDataset(CONFIG['VAL_IMG'], CONFIG['VAL_ANN'], CONFIG, train=False)
    
    model_label2coco_id = {v: k for k, v in train_dataset.catid2label.items()}
    logger.info(f"Model Label -> COCO ID Map: {model_label2coco_id}")

    # 2. Loaders
    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True,
        num_workers=CONFIG['NUM_WORKERS'], persistent_workers=True, pin_memory=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False,
        num_workers=CONFIG['NUM_WORKERS'], persistent_workers=True, pin_memory=True, collate_fn=collate_fn
    )

    # 3. Model
    model = build_model(CONFIG['NUM_MODEL_CLASSES'], device)
    
    # 4. Optimizer
    logger.info("Using AdamW Optimizer (Stability Mode)")
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=CONFIG['LEARNING_RATE'], 
        weight_decay=CONFIG['WEIGHT_DECAY']
    )
    
    # 5. Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG['EPOCHS'], eta_min=1e-6
    )
    
    # Scaler
    scaler = GradScaler() if CONFIG['USE_AMP'] else None

    # --- RESUME LOGIC ---
    start_epoch = 0
    best_val_map = 0.0
    
    if CONFIG.get('RESUME_TRAINING', False) and CONFIG.get('RESUME_CHECKPOINT'):
        resume_path = CONFIG['RESUME_CHECKPOINT']
        if os.path.exists(resume_path):
            logger.info(f" RESUMING TRAINING from: {resume_path}")
            try:
                checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
                model.load_state_dict(checkpoint['model_state_dict'])
                if 'optimizer_state_dict' in checkpoint:
                    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                start_epoch = checkpoint.get('epoch', 0) + 1
                best_val_map = checkpoint.get('best_val_map', 0.0)
                logger.info(f" Successfully resumed from Epoch {start_epoch}. Best mAP: {best_val_map:.4f}")
            except Exception as e:
                logger.error(f" FAILED to resume: {e}. Starting from scratch.")
        else:
            logger.warning(f" Checkpoint not found: {resume_path}. Starting from scratch.")
    else:
        logger.info(" Starting training from scratch (Epoch 0)")
    
    # --- TRAINING LOOP ---
    checkpoint_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'latest.pth')
    best_model_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'best.pth') # Path to save best model
    
    for epoch in range(start_epoch, CONFIG['EPOCHS']):
        logger.info(f"EPOCH {epoch+1}/{CONFIG['EPOCHS']}")
        
        metrics = train_one_epoch(model, optimizer, train_loader, device, epoch+1, scaler, scheduler, CONFIG, metrics_logger)
        
        if (epoch + 1) % CONFIG['VAL_FREQUENCY'] == 0:
            val_res = validate_model(
                model, val_loader, device, CONFIG, logger, 
                category_id_map=model_label2coco_id
            )
            metrics_logger.log_validation(epoch+1, val_res, val_res['inference_time'])
            
            if val_res['mAP'] > best_val_map:
                best_val_map = val_res['mAP']
                torch.save(model.state_dict(), best_model_path) # Save best model weights
                logger.info(f"*** NEW BEST mAP: {best_val_map:.4f} ***")

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_map': best_val_map,
            'config': CONFIG
        }, checkpoint_path)

    return {'best_map': best_val_map, 'final_epoch': CONFIG['EPOCHS'], 'device': device}

if __name__ == '__main__':
    try:
        result = main()
        device = result['device']
        
        # Define best model path (Output directory)
        best_model_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'best.pth')
        
        # --- 1. EXPORT MODEL (Optional) ---
        if CONFIG.get('EXPORT_MODEL', False):
            if os.path.exists(best_model_path):
                logger.info("Starting Export Process...")
                # Re-load best model to ensure clean state
                best_model = build_model(CONFIG['NUM_MODEL_CLASSES'], device)
                best_model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
                best_model.to(device)
                run_exports(model=best_model, device=device, config=CONFIG, logger=logger)
            else:
                logger.warning("Best model not found for export.")

        # --- 2. RUN TEST EVALUATION (Independent Logic) ---
        # Chạy độc lập, không phụ thuộc vào Export
        if CONFIG.get('RUN_TEST_EVAL', False):
            logger.info("=" * 80)
            logger.info(" STARTING AUTOMATIC TEST SET EVALUATION")
            logger.info("=" * 80)
            
            if os.path.exists(best_model_path):
                # Re-load best model (nếu chưa load ở bước export)
                # Nếu đã load ở trên rồi thì có thể dùng lại biến best_model, nhưng load lại cho chắc chắn an toàn bộ nhớ
                test_model = build_model(CONFIG['NUM_MODEL_CLASSES'], device)
                test_model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
                test_model.to(device)
                test_model.eval()
                
                evaluator = ComprehensiveEvaluator(
                    test_img_dir=CONFIG['TEST_IMG'], test_ann_file=CONFIG['TEST_ANN'],
                    logger=logger, config=CONFIG
                )
                evaluator.evaluate_on_test_set(
                    model=test_model, device=device, classnames=USER_CLASSES_LIST,
                    score_threshold=0.05 # Threshold thấp để đánh giá toàn diện
                )
            else:
                logger.error(f" Cannot run test eval: Best model not found at {best_model_path}")

        # Generate Plots
        generate_publication_figures(CONFIG, logger)
            
    except Exception as e:
        logger.error(f"CRITICAL ERROR: {e}")
        import traceback
        traceback.print_exc()